# Connect to Database

In [ ]:
import sqlalchemy
import pandas as pd
import numpy as np
from google.colab import userdata
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.ticker as mtick

In [ ]:
DB_USER = userdata.get('DB_USER')
DB_PASSWORD = userdata.get('DB_PASSWORD')
DB_HOST = userdata.get('DB_HOST')
DB_NAME = userdata.get('DB_NAME')
DB_PORT = "5432"

In [ ]:
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = sqlalchemy.create_engine(connection_string)

In [ ]:
pd.set_option('display.max_columns', None)

05/06

In [ ]:
all_data_so_far = """
SELECT
    timestamp,
    gateway_serial,
    "total_system_kWh",
    active_assets,
    active_asset_count,
    dominant_asset,
    line_to_neutral_voltage_phase_a,
    line_to_neutral_voltage_phase_b,
    line_to_neutral_voltage_phase_c,
    line_current_overall_phase_a,
    line_current_overall_phase_b,
    line_current_overall_phase_c,
    input_channel_1_current,
    input_channel_2_current,
    input_channel_3_current,
    input_channel_4_current,
    input_channel_5_current,
    input_channel_6_current
FROM public.smart_device_readings
ORDER BY timestamp DESC;
"""

all_data_df = pd.read_sql(all_data_so_far, engine)

In [ ]:
complete_data_so_far = """
SELECT
    timestamp,
    gateway_serial,
    "total_system_kWh",
    active_assets,
    active_asset_count,
    dominant_asset,
    line_to_neutral_voltage_phase_a,
    line_to_neutral_voltage_phase_b,
    line_to_neutral_voltage_phase_c,
    line_current_overall_phase_a,
    line_current_overall_phase_b,
    line_current_overall_phase_c,
    input_channel_1_current,
    input_channel_2_current,
    input_channel_3_current,
    input_channel_4_current,
    input_channel_5_current,
    input_channel_6_current
FROM public.smart_device_readings
WHERE gateway_serial IN ('EHM21120502', 'EHM54090515') AND timestamp >= '2026-05-20 00:00:00'
ORDER BY timestamp DESC;
"""

complete_df = pd.read_sql(complete_data_so_far, engine)

In [ ]:
# Finding all non-zero minimum

# 1. Define your target columns
current_cols = [
    "input_channel_1_current",
    "input_channel_2_current",
    "input_channel_3_current",
    "input_channel_4_current",
    "input_channel_5_current",
    "input_channel_6_current",
]

# 2. Subset, replace 0 with NaN, group, and find min
min_nonzero_df = (
    all_data_df[["gateway_serial"] + current_cols]
    .replace(0, np.nan)
    .groupby("gateway_serial")
    .min()
)

In [ ]:
min_nonzero_df.index

In [ ]:
min_nonzero_df.to_csv('minimum_current_per_site_all_time.csv')

In [ ]:
all_data_df[all_data_df['gateway_serial'] == 'EHM21120502'].input_channel_1_current.hist(bins=1)

In [ ]:
all_data_df[all_data_df['gateway_serial'] == 'EHM54090515'].input_channel_1_current.hist(bins=3)

In [ ]:
# Group your columns
ch_1_3 = ["input_channel_1_current", "input_channel_2_current", "input_channel_3_current"]
ch_4_6 = ["input_channel_4_current", "input_channel_5_current", "input_channel_6_current"]
all_channels = ch_1_3 + ch_4_6

In [ ]:
actual_all_data_df = all_data_df[all_data_df['gateway_serial'].isin(['EHM21120502', 'EHM54090515'])].copy()

In [ ]:
# Create a boolean mask where all 6 channels are not equal to 0
all_nonzero_mask = (complete_df[all_channels] != 0).all(axis=1)

# Filter your dataframe using the mask
filtered_df = complete_df[all_nonzero_mask]

In [ ]:
filtered_df.gateway_serial.value_counts()

In [ ]:
filtered_df.head()

In [ ]:
complete_df.sample(5)

When 1 - 3 are active and 4 - 6 are zero

In [ ]:
# Condition 1: Channels 1-3 are NOT zero
cond_1_3_nonzero = (complete_df[ch_1_3] != 0).all(axis=1)

# Condition 2: Channels 4-6 ARE zero
cond_4_6_zero = (complete_df[ch_4_6] == 0).any(axis=1)

# Combine both conditions using the bitwise AND (&) operator
specific_mask = cond_1_3_nonzero & cond_4_6_zero

# Filter your dataframe
filtered_df = complete_df[specific_mask]

In [ ]:
# 1. Define your two distinct groups
ch_1_3 = ["input_channel_1_current", "input_channel_2_current", "input_channel_3_current"]
ch_4_6 = ["input_channel_4_current", "input_channel_5_current", "input_channel_6_current"]

# 2. Identify true "active" channels (not 0 and not NaN) for both groups
ch_1_3_active = ~complete_df[ch_1_3].isin([0, np.nan])
ch_4_6_active = ~complete_df[ch_4_6].isin([0, np.nan])

# 3. Apply your criteria
# Condition 1: Group 1-3 has either 1, 2, or 3 active channels (at least one)
cond_1 = ch_1_3_active.sum(axis=1).isin([1, 2, 3])

# Condition 2: Group 4-6 has EXACTLY 1 active channel
cond_2 = ch_4_6_active.sum(axis=1) == 1

# 4. Combine both conditions and filter
final_mask = cond_1 & cond_2
filtered_df = complete_df[final_mask]

In [ ]:
filtered_df.gateway_serial.value_counts()

In [ ]:
filtered_df.active_assets.value_counts()

In [ ]:
grid_gen_aba_df = complete_df[(complete_df['gateway_serial'] == 'EHM21120502') & (complete_df['active_assets'] == 'Grid,Generator 1')].copy()

In [ ]:
# 1. Stack the dataframes together
combined = pd.concat([grid_gen_aba_df, filtered_df])

# 2. Drop all duplicates completely (keep=False removes BOTH copies of a duplicate)
unique_rows = combined.drop_duplicates(keep=False)

In [ ]:
unique_rows

In [ ]:
# Perform an outer merge with the indicator turned on
merged = grid_gen_aba_df.merge(filtered_df, how='outer', indicator=True)

# Filter out the rows that exist in 'both' dataframes
unique_rows = merged[merged['_merge'] != 'both']

In [ ]:
unique_rows

In [ ]:
unique_rows.input_channel_5_current

In [ ]:
filtered_df.columns

In [ ]:
summary_df_complete = complete_df[complete_df['gateway_serial'] == 'EHM21120502'][['input_channel_1_current', 'input_channel_2_current',
       'input_channel_3_current', 'input_channel_4_current',
       'input_channel_5_current', 'input_channel_6_current']].agg(['min', 'max', 'mean']).T

In [ ]:
summary_df

In [ ]:
summary_df_complete

In [ ]:
complete_df[(complete_df['gateway_serial'] == 'EHM21120502') & (complete_df['input_channel_5_current'] == 0)].active_assets.value_counts()

In [ ]:
complete_df[(complete_df['gateway_serial'] == 'EHM21120502') & (complete_df['input_channel_5_current'] == 0)]

In [ ]:
complete_df[(complete_df['gateway_serial'] == 'EHM21120502') & (complete_df['active_assets'] == 'Generator 1')][['input_channel_4_current',
       'input_channel_5_current', 'input_channel_6_current']]

In [ ]:
complete_df[(complete_df['gateway_serial'] == 'EHM21120502')].active_assets.value_counts()

In [ ]:
# 1. Define your two distinct groups
columns = ["input_channel_1_current", "input_channel_2_current", "input_channel_3_current", "input_channel_4_current", "input_channel_5_current", "input_channel_6_current"]

# Filter your dataframe
filtered_df_threshold_05 = complete_df[((complete_df[columns] < 0.5).any(axis=1)) & (complete_df['gateway_serial'] == 'EHM21120502')]

In [ ]:
# 1. Define your two distinct groups
columns = ["input_channel_1_current", "input_channel_2_current", "input_channel_3_current", "input_channel_4_current", "input_channel_5_current", "input_channel_6_current"]

# Filter your dataframe
filtered_df_threshold_05 = complete_df[((complete_df[columns] < 0.5).any(axis=1)) & ((complete_df[columns] != 0).all(axis=1))]

In [ ]:
filtered_df_threshold_05.gateway_serial.value_counts()

In [ ]:
filtered_df_threshold_05.sample(5)

In [ ]:
all_channels

In [ ]:


# 1. Get the maximum value across the 6 channels for each row
row_max = complete_df[all_channels].max(axis=1)

# 2. Create a mask where the maximum is greater than 0 BUT less than 0.5
low_activity_mask = (row_max > 0) & (row_max < 0.5)

# 3. Filter the dataframe
filtered_df = complete_df[low_activity_mask]

In [ ]:
filtered_df

In [ ]:
complete_df[(complete_df['input_channel_1_current'] < 0.5) & (complete_df['input_channel_1_current'] > 0)]

In [ ]:
df[df['active_assets'] == 'Grid,Generator 1'][['grid_active', 'gen_active']].value_counts()

In [ ]:
df[df['active_assets'] == 'Grid,Generator 1']['grid_active'].value_counts()

In [ ]:
(df['grid_active'] + df['gen_active']).value_counts()

In [ ]:
len(df[df['total_system_kWh'] == 0])

In [ ]:
total_kWh_0 = df[df['total_system_kWh'] == 0]

In [ ]:
failed_attempt = df.loc[~(df['grid_active'] | df['gen_active'])]

In [ ]:
failed_attempt = df.loc[~(df['grid_active'] | df['gen_active']) & (df['total_system_kWh'] != 0), ['timestamp', 'active_power_overall_total',
    'apparent_power_overall_total',
    'power_factor_overall',
    'load',
    'frequency','total_system_kWh', 'gateway_serial', 'input_channel_1_current',
 'input_channel_2_current',
 'input_channel_3_current',
 'input_channel_4_current',
 'input_channel_5_current',
 'input_channel_6_current', 'grid_active', 'gen_active']]

In [ ]:
# Perform an outer merge with the indicator turned on
merged = failed_attempt.merge(total_kWh_0, how='outer', indicator=True)

# Filter out the rows that exist in 'both' dataframes
unique_rows = merged[merged['_merge'] != 'both']

In [ ]:
unique_rows

In [ ]:
unique_rows.to_csv('examine_outside_cases.csv')

08/06

In [ ]:
# Updated df

updated_complete_data_so_far = """
SELECT
    timestamp,
    gateway_serial,
    active_power_overall_total,
    apparent_power_overall_total,
    power_factor_overall,
    "total_system_kWh",
    load,
    frequency,
    active_assets,
    active_asset_count,
    dominant_asset,
    line_to_neutral_voltage_phase_a,
    line_to_neutral_voltage_phase_b,
    line_to_neutral_voltage_phase_c,
    line_current_overall_phase_a,
    line_current_overall_phase_b,
    line_current_overall_phase_c,
    input_channel_1_current,
    input_channel_2_current,
    input_channel_3_current,
    input_channel_4_current,
    input_channel_5_current,
    input_channel_6_current
FROM public.smart_device_readings
WHERE gateway_serial IN ('EHM21120502', 'EHM54090515') AND timestamp >= '2026-05-20 00:00:00'
ORDER BY timestamp DESC;
"""

updated_complete_df = pd.read_sql(updated_complete_data_so_far, engine)

In [ ]:
len(updated_complete_df)

In [ ]:
updated_complete_df[(updated_complete_df['timestamp'] == pd.Timestamp('2026-05-23 14:50:09+00:00')) & (updated_complete_df['gateway_serial'] == 'EHM54090515')]

In [ ]:
# 1. Define your center timestamp
target_time = pd.Timestamp('2026-05-23 14:50:09+00:00')

# 2. Define the 5-minute bounds
start_time = target_time - pd.Timedelta(minutes=5)
end_time = target_time + pd.Timedelta(minutes=5)

# 3. Filter the dataframe using .between()
filtered_df = updated_complete_df[
    (updated_complete_df['timestamp'].between(start_time, end_time)) &
    (updated_complete_df['gateway_serial'] == 'EHM54090515')
]

In [ ]:
filtered_df

Continuing with the current threshold work

In [ ]:
df = updated_complete_df.copy()

In [ ]:
CURRENT_THRESHOLD = 0.5

grid_phases = ['input_channel_1_current', 'input_channel_2_current', 'input_channel_3_current']
gen_phases  = ['input_channel_4_current', 'input_channel_5_current', 'input_channel_6_current']

voltage_cols = [
    'line_to_neutral_voltage_phase_a',
    'line_to_neutral_voltage_phase_b',
    'line_to_neutral_voltage_phase_c'
]

def asset_active(df, phase_cols, voltage_cols, threshold=CURRENT_THRESHOLD):
    max_current = df[phase_cols].max(axis=1)
    voltage_present = (df[voltage_cols].abs() > 0).any(axis=1)
    return (max_current > threshold) & voltage_present

df['grid_active'] = asset_active(df, grid_phases, voltage_cols)
df['gen_active']  = asset_active(df, gen_phases,  voltage_cols)

In [ ]:
# 2. Define the conditions for the new column
conditions = [
    (df['grid_active'] & df['gen_active']),  # Both are active (overlap/error state)
    df['grid_active'],                       # Only Grid is active
    df['gen_active']                         # Only Generator is active
]

# 3. Define the matching labels for those conditions
choices = ['Both', 'Grid', 'Generator 1']

# 4. Create the new column (default handles when both are False)
df['active_asset'] = np.select(conditions, choices, default='None')

In [ ]:
df.columns

In [ ]:
df[['active_assets', 'active_asset']]

In [ ]:
None == ''

July Data

In [ ]:
complete_data_so_far = """
SELECT
    timestamp,
    gateway_serial,
    active_power_overall_total,
    apparent_power_overall_total,
    power_factor_overall,
    "total_system_kWh",
    load,
    frequency,
    active_assets,
    active_asset_count,
    workhour,
    transformer_capacity,
    transformer_load_percentage,
    line_to_neutral_voltage_phase_a,
    line_to_neutral_voltage_phase_b,
    line_to_neutral_voltage_phase_c,
    line_current_overall_phase_a,
    line_current_overall_phase_b,
    line_current_overall_phase_c,
    voltage_unbalance_factor,
    current_unbalance_factor,
    total_harmonic_distortion_current_phase_a,
    total_harmonic_distortion_current_phase_b,
    total_harmonic_distortion_current_phase_c
FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515' AND timestamp >= '2026-06-01 00:00:00'
ORDER BY timestamp DESC;
"""

complete_df = pd.read_sql(complete_data_so_far, engine)

In [ ]:
complete_df_copy = complete_df.copy()

In [ ]:
# 1. Convert the column to datetime (just in case it loaded as a string/object)
complete_df_copy["timestamp"] = pd.to_datetime(complete_df_copy["timestamp"])

# 2. Set the column as the index
complete_df_copy = complete_df_copy.set_index("timestamp")

# 3. Crucial: Sort the index chronologically
complete_df_copy = complete_df_copy.sort_index()

In [ ]:
df_filled = complete_df_copy.resample("1min").first()

In [ ]:
df_filled.isna().sum()

In [ ]:
complete_df_copy.sample(10)

In [ ]:
# turn 'timestamp' into a regular column
df = df_filled.reset_index()

# Rename index into timestamp if not currently nameed timestamp
if "timestamp" not in df.columns and "index" in df.columns:
    df = df.rename(columns={"index": "timestamp"})

# Clean, sort, and parse data
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Replace empty strings and NaNs with clear tracking labels
df["active_assets"] = df["active_assets"].replace(
    {"": "None / Standby", np.nan: "No Power"}
)

# Fill NaN load values with 0 to show a visible line during outages
df["load"] = df["load"].fillna(0)

In [ ]:
df_copy_for_working_hours = df.copy()
df_copy_for_working_hours["hour"] = df_copy_for_working_hours["timestamp"].dt.hour
df_copy_for_working_hours["date"] = df_copy_for_working_hours["timestamp"].dt.date
df_copy_for_working_hours["day_of_week"] = df_copy_for_working_hours["timestamp"].dt.dayofweek # Monday=0, Sunday=6

# Filter for working hours (8 AM to 5 PM inclusive, meaning hour >= 8 and hour < 18)
df_working_hours = df_copy_for_working_hours[
    (df_copy_for_working_hours["hour"] >= 8) &
    (df_copy_for_working_hours["hour"] < 18) &
    (df_copy_for_working_hours["day_of_week"] >= 0) & # Monday
    (df_copy_for_working_hours["day_of_week"] <= 4)   # Friday
].copy()

In [ ]:
# Group by date and active_assets, then count the number of 1-minute intervals (duration)
daily_duration = (
    df_working_hours.groupby(["date", "active_assets"])
    .size()
    .reset_index(name="duration_minutes")
)

# Pivot the data so dates are rows and assets are individual columns
pivot_df_duration = daily_duration.pivot(
    index="date", columns="active_assets", values="duration_minutes"
).fillna(0)

# Define the order for plotting, including 'No Power'
hue_order_duration = ["Grid", "Generator 1", "No Power"]

available_cols_duration = [
    col for col in hue_order_duration if col in pivot_df_duration.columns
]
pivot_df_duration = pivot_df_duration[available_cols_duration]

# Apply custom color palette including 'No Power'
custom_palette_duration = {
    "Grid": "#1f77b4",  # Blue
    "Generator 1": "#ff7f0e",  # Orange
    "No Power": "#d62728",  # Red for no power
    "None / Standby": "#9467bd",  # Purple
    "Both": "#8c564b",  # Brown
}
colors_duration = [
    custom_palette_duration[col] for col in pivot_df_duration.columns
]

# Generate the Stacked Bar Plot for duration
fig_duration, ax_duration = plt.subplots(figsize=(12, 6))

# Format the date index strings to look clean on the X-axis
pivot_df_duration.index = pd.to_datetime(pivot_df_duration.index).strftime("%a, %b %d")

pivot_df_duration.plot(
    kind="bar",
    stacked=True,
    color=colors_duration,
    width=0.6,
    edgecolor="black",
    linewidth=0.8,
    alpha=0.85,
    ax=ax_duration,
)

# --- HIDE 0 VALUES COMPLETELY ---
for container in ax_duration.containers:
    labels = [
        f"{v.get_height():,.0f}"
        if v.get_height() > 0
        else ""
        for v in container
    ]

    ax_duration.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=9,
        fontweight="bold",
        color="white",
    )
# ------------------------------------------

# --- INTEGRATED: ADD TOTAL DURATION LABELS ON TOP OF BARS ---
daily_totals_duration = pivot_df_duration.sum(axis=1)
max_daily_total_duration = daily_totals_duration.max()

for i, total in enumerate(daily_totals_duration):
    if total > 0:
        ax_duration.text(
            i,
            total + (max_daily_total_duration * 0.015),
            f"{total:,.0f}",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
            color="#333333",
        )
# --------------------------------------------------------------

# Descriptive labels and text formatting
ax_duration.set_title(
    "Daily Active Asset Duration During Working Hours (8 AM - 5 PM) - Weekdays Only",
    fontsize=14,
    pad=15,
    weight="bold",
)
ax_duration.set_xlabel("Date", fontsize=11, labelpad=10)
ax_duration.set_ylabel("Total Duration (Minutes)", fontsize=11, labelpad=10)

# Keep X-axis labels completely straight
plt.xticks(rotation=0, ha="center")

# Remove the top and right borders (spines)
ax_duration.spines["top"].set_visible(False)
ax_duration.spines["right"].set_visible(False)

# Clean up the legend layout
ax_duration.legend(
    title="System State",
    loc="upper right",
    frameon=True,
    facecolor="white",
    edgecolor="none",
)

plt.tight_layout()

# Save the visualization
start_str_duration = pd.to_datetime(daily_duration["date"].min()).strftime("%Y-%m-%d")
end_str_duration = pd.to_datetime(daily_duration["date"].max()).strftime("%Y-%m-%d")
plt.savefig(
    f"Daily_Active_Asset_Duration_Working_Hours_Weekdays_{start_str_duration}_to_{end_str_duration}.png",
    dpi=300,
)

plt.show()

In [ ]:
complete_df

In [ ]:
df = complete_df.copy()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Ensure timestamp structure and extract the calendar date
"""
df = abmf_hoffice_24_30_df.reset_index()
if "timestamp" not in df.columns and "index" in df.columns:
    df = df.rename(columns={"index": "timestamp"})
"""

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["date"] = df["timestamp"].dt.date

# 2. Handle asset labels consistently
df["active_assets"] = df["active_assets"].replace(
    {"": "None / Standby", np.nan: "No Power"}
)

# 3. Group by date and asset, then sum the total kWh consumption
daily_energy = (
    df.groupby(["date", "active_assets"])["total_system_kWh"]
    .sum()
    .reset_index()
)

# 4. Pivot the data so dates are rows and assets are individual columns
pivot_df = daily_energy.pivot(
    index="date", columns="active_assets", values="total_system_kWh"
).fillna(0)

# Include ONLY Grid and Generator 1
hue_order = ["Grid", "Generator 1"]
available_cols = [col for col in hue_order if col in pivot_df.columns]
pivot_df = pivot_df[available_cols]

# Apply your unified corporate color palette
custom_palette = {
    "Grid": "#1f77b4",          # Blue
    "Generator 1": "#ff7f0e",   # Orange
}
colors = [custom_palette[col] for col in pivot_df.columns]

# 5. Generate the Stacked Bar Plot
fig, ax = plt.subplots(figsize=(12, 6))

# Format the date index strings to look clean on the X-axis (e.g., "Mon, Oct 24")
pivot_df.index = pd.to_datetime(pivot_df.index).strftime("%a, %b %d")

pivot_df.plot(
    kind="bar",
    stacked=True,
    color=colors,
    width=0.6,
    edgecolor="black",
    linewidth=0.8,
    alpha=0.85,
    ax=ax,
)

# --- HIDE 0 VALUES COMPLETELY ---
for container in ax.containers:
    # Only format the string if the height is strictly greater than 0
    labels = [f"{v.get_height():,.0f}" if v.get_height() > 0 else "" for v in container]

    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=9,
        fontweight="bold",
        color="white"
    )
# ------------------------------------------

# --- INTEGRATED: ADD TOTAL ENERGY SUM LABELS ON TOP OF BARS ---
# Calculate the combined daily totals and find the maximum peak for scaling the label buffer
daily_totals = pivot_df.sum(axis=1)
max_daily_total = daily_totals.max()

for i, total in enumerate(daily_totals):
    if total > 0:
        ax.text(
            i,
            total + (max_daily_total * 0.015), # 1.5% vertical padding cushion above the bar edge
            f"{total:,.0f}",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
            color="#333333"                     # Premium dark gray color for clear readability
        )
# --------------------------------------------------------------

# Descriptive labels and text formatting
ax.set_title(
    "Daily Energy Consumption Profile by Power Source",
    fontsize=14,
    pad=15,
    weight="bold",
)
ax.set_xlabel("Date", fontsize=11, labelpad=10)
ax.set_ylabel("Total Energy Consumed (kWh)", fontsize=11, labelpad=10)

# Keep X-axis labels completely straight
plt.xticks(rotation=0, ha="center")

# Remove the top and right borders (spines)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Clean up the legend layout
ax.legend(
    title="System State",
    loc="upper right",
    frameon=True,
    facecolor="white",
    edgecolor="none",
)

plt.tight_layout()

# Save the visualization
start_str = pd.to_datetime(daily_energy["date"].min()).strftime("%Y-%m-%d")
end_str = pd.to_datetime(daily_energy["date"].max()).strftime("%Y-%m-%d")
plt.savefig(f"Daily_Energy_Stacked_Source_{start_str}_to_{end_str}.png", dpi=300)

plt.show()